# Thai Election OCR Agentic Pipeline

Notebook นี้ประมวลผลภาพสแกนเอกสารผลการเลือกตั้งไทย โดยใช้ EasyOCR สกัดข้อความจากตารางในแต่ละหน้า แล้วจับคู่คะแนนเสียงกับพรรคการเมืองตาม template ที่กำหนด

**ลำดับขั้นตอนหลักของ pipeline:**

1. อ่านไฟล์ภาพแต่ละหน้าโดยตรงจาก `data/images/`
2. ตรวจหาตำแหน่งตารางในภาพด้วย OpenCV แล้วส่งเฉพาะส่วนนั้นให้ EasyOCR
3. บันทึก OCR cache ต่อหน้าไว้ที่ `data/ocr_cache_easyocr/`
4. แปลงตัวเลขไทยและ OCR noise เป็นตัวเลขอารบิก
5. จำแนกประเภทหน้า (table / mixed / header / summary / other)
6. รวมผลจากทุกหน้าของแต่ละเอกสาร แล้ว map `party -> votes`
7. เขียนผลลัพธ์เป็น `data/submission.csv` ในรูปแบบ `id,votes`

**โครงสร้างไดเรกทอรีที่จำเป็น:**

```
data/
  images/                   # ไฟล์ภาพ .png (ชื่อ <doc_id>_page<N>.png)
  sample_labels/            # JSON labels ตัวอย่างสำหรับดึงชื่อจังหวัด
  submission_template.csv   # template ระบุ id, doc_id, row_num, party_name
  submission_template_v3.csv# scaffold สำหรับสร้าง submission
  labels/                   # (สร้างอัตโนมัติ) JSON label ต่อหน้า
  ocr_cache_easyocr/        # (สร้างอัตโนมัติ) markdown cache ต่อหน้า
```

**การรัน notebook:** สามารถรันแบบแยกขั้นตอนได้ โดยควบคุมผ่าน flag `RUN_SCAN_TO_LABELS` และ `RUN_BUILD_SUBMISSION` ใน cell Config (Section 2)


## 1. Setup

**วัตถุประสงค์:** ตรวจสอบและติดตั้ง dependency เบื้องต้นที่จำเป็นก่อนรัน pipeline

**สิ่งที่ cell นี้ทำ:**
- นิยามฟังก์ชัน `ensure_package()` สำหรับติดตั้ง package ผ่าน `pip` เมื่อยังไม่มีในสภาพแวดล้อม
- ติดตั้ง `tqdm` (progress bar) ซึ่งใช้ตลอดทั้ง notebook
- `easyocr` จะถูกติดตั้งต่อมาเฉพาะเมื่อ scan stage ทำงานจริง เพื่อประหยัดเวลาหาก run เฉพาะ submission stage

**หมายเหตุ:** ควรรัน cell นี้ก่อนเสมอ ทุกครั้งที่เริ่ม session ใหม่


In [267]:
from __future__ import annotations

import importlib.util
import subprocess
import sys


def ensure_package(package: str, import_name: str | None = None) -> None:
    module_name = import_name or package.replace('-', '_')
    if importlib.util.find_spec(module_name) is None:
        print(f'Installing {package} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])


ensure_package('tqdm')
print('Ready: tqdm installed or already available')

Ready: tqdm installed or already available


## 2. Config

**วัตถุประสงค์:** กำหนดค่าพารามิเตอร์และ path ทั้งหมดที่ใช้ใน pipeline รวมถึงโหลด library หลักและนิยาม dataclass สำหรับ row data

**พารามิเตอร์ที่ควรปรับก่อนรัน:**

| พารามิเตอร์ | ค่าเริ่มต้น | คำอธิบาย |
|---|---|---|
| `RUN_SCAN_TO_LABELS` | `False` | ตั้งเป็น `True` เพื่อรัน OCR scan stage |
| `RUN_BUILD_SUBMISSION` | `True` | ตั้งเป็น `True` เพื่อสร้าง submission.csv |
| `EASYOCR_USE_GPU` | `True` | ปิด GPU หากสภาพแวดล้อมไม่มี CUDA |
| `RESCAN_EXISTING_LABELS` | `False` | `True` = OCR ใหม่ทับไฟล์ที่มีอยู่แล้ว |
| `FORCE_RERUN_OCR` | `False` | `True` = บังคับ re-run OCR ทุกหน้า |
| `LIMIT_DOCS` | `None` | จำกัดจำนวนเอกสาร เช่น `10` สำหรับทดสอบ |
| `EASYOCR_CONFIDENCE_THRESHOLD` | `0.10` | ตัด OCR result ที่ confidence ต่ำกว่าค่านี้ |

**Dataclass ที่นิยามใน cell นี้:**
- `TemplateRow`: ข้อมูลแต่ละแถวใน submission template (id, doc_id, row_num, party_name)
- `ExtractedRow`: ผลลัพธ์ OCR ที่สกัดได้จากหน้าเอกสาร (party_name, votes_text และ metadata)

**หมายเหตุ:** cell นี้ยังโหลด `.env` file อัตโนมัติหากมีอยู่ในโฟลเดอร์เดียวกัน และสร้างไดเรกทอรี `labels/` และ `ocr_cache_easyocr/` หากยังไม่มี


In [268]:
import csv
import importlib.util
import json
import os
import re
import subprocess
import sys
import time
import unicodedata
from collections import defaultdict
from dataclasses import dataclass
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

from PIL import Image
from tqdm.auto import tqdm

DATA_ROOT = Path('data')
IMAGE_DIR = DATA_ROOT / 'images'
IMAGE_CUT_DIR = DATA_ROOT / 'image_cut'
SAMPLE_LABEL_DIR = DATA_ROOT / 'sample_labels'
TEMPLATE_PATH = DATA_ROOT / 'submission_template.csv'
if not TEMPLATE_PATH.exists():
    fallback_template_path = DATA_ROOT / 'submission_template copy.csv'
    if fallback_template_path.exists():
        TEMPLATE_PATH = fallback_template_path
SUBMISSION_SCAFFOLD_PATH = DATA_ROOT / 'submission_template_v3.csv'
LABEL_DIR = DATA_ROOT / 'labels'
OCR_CACHE_DIR = DATA_ROOT / 'ocr_cache_easyocr'
OUTPUT_PATH = DATA_ROOT / 'submission.csv'
PARTIAL_OUTPUT_PATH = DATA_ROOT / 'submission.partial.csv'
ENV_PATH = Path('.env')

EASYOCR_LANGUAGES = ['th', 'en']
EASYOCR_USE_GPU = True
EASYOCR_CONFIDENCE_THRESHOLD = 0.10
EASYOCR_MAG_RATIO = 1.2
EASYOCR_CANVAS_SIZE = 2560
MIN_INTERVAL_SECONDS = 0.0
PAUSE_EVERY_PAGES = 0
PAUSE_SECONDS = 0
MAX_RETRIES = 2
RETRY_BASE_SECONDS = 5
TABLE_MIN_REGION_AREA_RATIO = 0.0025
TABLE_MIN_WIDTH_RATIO = 0.18
TABLE_MIN_HEIGHT_RATIO = 0.035
TABLE_MIN_LINE_DENSITY = 0.015
TABLE_FALLBACK_DENSITY = 0.0025
TABLE_DETECTION_MAX_SIDE = 1800
TABLE_REGION_PADDING = 16
RESCAN_EXISTING_LABELS = False
FORCE_RERUN_OCR = False
REBUILD_IMAGE_CUT = False
STORE_MARKDOWN_IN_LABELS = True
ALLOW_MISSING_LABELS = False
RUN_PREPARE_IMAGE_CUT = False
RUN_SCAN_TO_LABELS = False
RUN_BUILD_SUBMISSION = False
LIMIT_DOCS: Optional[int] = None

assert IMAGE_DIR.exists(), f'Missing image directory: {IMAGE_DIR}'
assert TEMPLATE_PATH.exists(), f'Missing template file: {TEMPLATE_PATH}'
assert SUBMISSION_SCAFFOLD_PATH.exists(), f'Missing submission scaffold file: {SUBMISSION_SCAFFOLD_PATH}'
LABEL_DIR.mkdir(parents=True, exist_ok=True)
OCR_CACHE_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_CUT_DIR.mkdir(parents=True, exist_ok=True)


def load_dotenv_file(path: Path = ENV_PATH) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, value)


load_dotenv_file()
print(f'EasyOCR backend selected | languages={EASYOCR_LANGUAGES} | gpu={EASYOCR_USE_GPU}')
print(
    f'RUN_PREPARE_IMAGE_CUT={RUN_PREPARE_IMAGE_CUT}, '
    f'RUN_SCAN_TO_LABELS={RUN_SCAN_TO_LABELS}, RUN_BUILD_SUBMISSION={RUN_BUILD_SUBMISSION}, LIMIT_DOCS={LIMIT_DOCS}'
)


@dataclass(frozen=True)
class TemplateRow:
    id: str
    doc_id: str
    row_num: int
    party_name: str


@dataclass
class ExtractedRow:
    party_name: str
    votes_text: str
    source_page: Optional[int] = None
    candidate_name: Optional[str] = None
    rank_hint: Optional[int] = None
    raw_line: Optional[str] = None
    match_score: Optional[float] = None



EasyOCR backend selected | languages=['th', 'en'] | gpu=True
RUN_SCAN_TO_LABELS=False, RUN_BUILD_SUBMISSION=False, LIMIT_DOCS=None


## 3. Load Competition Structure

**วัตถุประสงค์:** โหลดโครงสร้างข้อมูลของการแข่งขัน ได้แก่ index ของไฟล์ภาพ, template การ submit และ metadata จังหวัด

**ฟังก์ชันที่นิยามใน cell นี้:**

- `parse_page_record(path)`: แยก `doc_id` และ `page_number` จากชื่อไฟล์ภาพ เช่น `constituency_10_1_page2.png` -> `(constituency_10_1, 2)`
- `build_document_index(image_dir)`: สแกนไดเรกทอรีภาพ จัดกลุ่มไฟล์ตาม doc_id และเรียงตามเลขหน้า
- `load_template(path)`: อ่าน `submission_template.csv` เป็น list ของ `TemplateRow`
- `group_template_by_doc(rows)`: จัดกลุ่ม template rows ตาม doc_id
- `parse_doc_id(doc_id)`: แยก doc_id เป็น type (party_list / constituency), province_code และ constituency_number
- `build_known_province_names(sample_dir)`: อ่านชื่อจังหวัดจาก sample labels JSON

**Output ที่ได้หลัง cell รัน:**
- `document_index`: dict ของ `{doc_id -> [page_path, ...]}`
- `template_by_doc`: dict ของ `{doc_id -> [TemplateRow, ...]}`
- `doc_ids`: list ของ doc_id ทั้งหมดที่จะประมวลผล (จำกัดตาม `LIMIT_DOCS`)
- `known_province_names`: dict ของ `{province_code -> province_name}`


In [269]:
PAGE_RE = re.compile(r'^(?P<doc_id>.+?)(?:_page(?P<page>\d+))?\.png$')


def parse_page_record(path: Path) -> Tuple[str, int]:
    match = PAGE_RE.match(path.name)
    if not match:
        raise ValueError(f'Unexpected image name: {path.name}')
    doc_id = match.group('doc_id')
    page_number = int(match.group('page') or '1')
    return doc_id, page_number


def build_document_index(image_dir: Path) -> Dict[str, List[Path]]:
    grouped: Dict[str, List[Tuple[int, Path]]] = defaultdict(list)
    for path in sorted(image_dir.glob('*.png')):
        doc_id, page_number = parse_page_record(path)
        grouped[doc_id].append((page_number, path))
    return {doc_id: [path for page_number, path in sorted(items)] for doc_id, items in grouped.items()}


def load_template(path: Path) -> List[TemplateRow]:
    rows: List[TemplateRow] = []
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(
                TemplateRow(
                    id=row['id'],
                    doc_id=row['doc_id'],
                    row_num=int(row['row_num']),
                    party_name=row['party_name'],
                )
            )
    return rows


def group_template_by_doc(rows: Sequence[TemplateRow]) -> Dict[str, List[TemplateRow]]:
    grouped: Dict[str, List[TemplateRow]] = defaultdict(list)
    for row in rows:
        grouped[row.doc_id].append(row)
    for doc_id in grouped:
        grouped[doc_id].sort(key=lambda row: row.row_num)
    return dict(grouped)


def parse_doc_id(doc_id: str) -> Dict[str, Any]:
    parts = doc_id.split('_')
    if parts[:2] == ['party', 'list'] and len(parts) >= 4:
        return {'type': 'party_list', 'province_code': parts[2], 'constituency_number': int(parts[3])}
    if parts[0] == 'constituency' and len(parts) >= 3:
        return {'type': 'constituency', 'province_code': parts[1], 'constituency_number': int(parts[2])}
    raise ValueError(f'Unexpected doc_id format: {doc_id}')


def build_known_province_names(sample_dir: Path) -> Dict[str, Optional[str]]:
    province_names: Dict[str, Optional[str]] = {}
    for path in sorted(sample_dir.glob('*.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        province_code = str(payload.get('province_code')) if payload.get('province_code') is not None else None
        if province_code:
            province_names[province_code] = payload.get('province_name')
    return province_names


document_index = build_document_index(IMAGE_DIR)
template_rows = load_template(TEMPLATE_PATH)
template_by_doc = group_template_by_doc(template_rows)
known_province_names = build_known_province_names(SAMPLE_LABEL_DIR)

doc_ids = sorted(template_by_doc)
if LIMIT_DOCS:
    doc_ids = doc_ids[:LIMIT_DOCS]

print(f'Documents in images: {len(document_index)}')
print(f'Documents in template: {len(template_by_doc)}')
print(f'Template rows: {len(template_rows)}')
print(f'Known province names from sample labels: {len(known_province_names)}')
print(f'Active documents for this run: {len(doc_ids)}')

Documents in images: 300
Documents in template: 300
Template rows: 10053
Known province names from sample labels: 4
Active documents for this run: 300


## 4. Text Normalization, Page Classification, and Mapping

**วัตถุประสงค์:** นิยามฟังก์ชัน utility ทั้งหมดสำหรับทำความสะอาดข้อความ, จำแนกหน้าเอกสาร, จับคู่ชื่อพรรค และแปลงผลลัพธ์เป็น submission format

**กลุ่มฟังก์ชันหลัก:**

**Text Normalization**
- `normalize_text()`: Unicode NFKC normalization + collapse whitespace
- `normalize_vote_text()`: แปลงตัวเลขไทย (๐-๙) เป็นอารบิก + แก้ไข OCR confusion เช่น `O->0`, `l->1`, `S->5`
- `normalize_party_key()`: normalize ชื่อพรรคสำหรับใช้เปรียบเทียบ (ลบ space, ลบ punctuation, casefold)

**Party Matching**
- `similarity(a, b)`: คำนวณความคล้ายคลึงของสองสตริงด้วย `SequenceMatcher`
- `best_party_match()`: หาชื่อพรรคที่ใกล้เคียงที่สุดจาก allowed list โดยมี threshold 0.55
- `choose_best_vote()`: เลือก votes string ที่ดีที่สุดระหว่างสองค่า (เลือกที่มี digit ยาวกว่า)

**Row Parsing**
- `parse_markdown_rows()`: แยก party และ votes จากแต่ละบรรทัดของ OCR markdown output
- `extract_vote_candidate()`: สกัด digit candidate จาก text segment
- `extract_rank_hint()`: ดึงเลขลำดับจากต้น line (สำหรับเรียงลำดับพรรค)

**Page Classification**
- `classify_page_content()`: ให้คะแนน signal และตัดสินใจว่าหน้านั้นเป็น `table`, `mixed`, `header`, `summary` หรือ `other` โดยรวม heuristic จาก image-based table detection, party hits, numeric hits และ summary keywords

**Result Mapping**
- `map_rows_to_template()`: map extracted rows เข้า template โดย direct lookup ก่อน ถ้าไม่ตรงจึง fuzzy match
- `build_result_items()`: สร้าง result list สุดท้ายพร้อม candidate name และ rank hint
- `write_submission()`: เขียน `id,votes` CSV


In [270]:
THAI_DIGITS = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
COMMON_DIGIT_CONFUSIONS = {
    'O': '0', 'o': '0', 'Q': '0', 'D': '0', 'U': '0',
    'I': '1', 'l': '1', '|': '1', '!': '1',
    'Z': '2',
    'S': '5', 's': '5',
    'B': '8',
    ',': '', '.': '', ' ': '', '-': '', '_': '', '/': '',
}
SUMMARY_KEYWORDS = [
    '????????????????????????',
    '??????????????????',
    '???????????????????',
    '??????',
    '????????',
    '?????????????????????????',
    '?????????????',
]


def normalize_text(text: Optional[str]) -> str:
    if text is None:
        return ''
    text = unicodedata.normalize('NFKC', str(text))
    text = text.replace('​', ' ').replace(' ', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def normalize_markdown_text(text: Optional[str]) -> str:
    if text is None:
        return ''
    text = unicodedata.normalize('NFKC', str(text)).replace('\r\n', '\n').replace('\r', '\n')
    normalized_lines = [line.rstrip() for line in text.split('\n')]
    return '\n'.join(normalized_lines).strip()




def normalize_party_key(text: Optional[str]) -> str:
    text = normalize_text(text).replace(' ', '')
    text = re.sub(r'[^\w฀-๿]', '', text)
    return text.casefold()


def normalize_vote_text(text: Optional[str]) -> str:
    text = normalize_text(text).translate(THAI_DIGITS)
    for src, dst in COMMON_DIGIT_CONFUSIONS.items():
        text = text.replace(src, dst)
    text = re.sub(r'[^0-9]', '', text)
    return text


def coerce_votes(text: Optional[str], default: str = '0') -> str:
    digits = normalize_vote_text(text)
    return digits if digits else default


def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize_party_key(a), normalize_party_key(b)).ratio()


def best_party_match(observed_party: str, allowed_parties: Sequence[str], threshold: float = 0.55) -> Tuple[Optional[str], float]:
    if not allowed_parties:
        return None, 0.0
    scored = sorted(((party, similarity(observed_party, party)) for party in allowed_parties), key=lambda item: item[1], reverse=True)
    best_party, best_score = scored[0]
    if best_score < threshold:
        return None, best_score
    return best_party, best_score


def choose_best_vote(existing: Optional[str], incoming: Optional[str]) -> str:
    existing_digits = normalize_vote_text(existing)
    incoming_digits = normalize_vote_text(incoming)
    if incoming_digits and not existing_digits:
        return incoming_digits
    if existing_digits and not incoming_digits:
        return existing_digits
    if len(incoming_digits) > len(existing_digits):
        return incoming_digits
    return existing_digits or incoming_digits or '0'


def template_lookup(rows: Sequence[TemplateRow]) -> Dict[str, TemplateRow]:
    return {normalize_party_key(row.party_name): row for row in rows}


def iter_markdown_lines(markdown_text: str) -> Iterable[str]:
    for raw_line in normalize_markdown_text(markdown_text).splitlines():
        line = normalize_text(raw_line.strip())
        if not line:
            continue
        if re.fullmatch(r'\|?\s*:?-{3,}:?\s*(\|\s*:?-{3,}:?\s*)+\|?', line):
            continue
        line = line.strip('|')
        parts = [normalize_text(part) for part in line.split('|') if normalize_text(part)]
        if parts:
            yield ' | '.join(parts)
        else:
            yield line


def extract_vote_candidate(text: str) -> str:
    matches = re.findall(r'[0-9๐-๙OolIQDSB,._\- ]{1,20}', text)
    best = ''
    for match in matches:
        digits = normalize_vote_text(match)
        if len(digits) >= len(best):
            best = digits
    return best


def extract_rank_hint(text: str) -> Optional[int]:
    match = re.match(r'^\D{0,4}(\d{1,3})\D', normalize_text(text))
    if not match:
        return None
    return int(match.group(1))


def extract_candidate_name(text: str, matched_party: Optional[str], votes_text: str) -> Optional[str]:
    candidate = normalize_text(text)
    if matched_party:
        candidate = candidate.replace(matched_party, ' ')
    if votes_text:
        candidate = candidate.replace(votes_text, ' ')
    candidate = re.sub(r'\b\d{1,3}\b', ' ', candidate)
    candidate = normalize_text(candidate.replace('|', ' '))
    return candidate or None


def serialize_extracted_row(row: ExtractedRow) -> Dict[str, Any]:
    return {
        'party': row.party_name,
        'votes': int(coerce_votes(row.votes_text)),
        'source_page': row.source_page,
        'candidate_name': row.candidate_name,
        'rank_hint': row.rank_hint,
        'raw_line': row.raw_line,
        'match_score': row.match_score,
    }


def deserialize_extracted_row(item: Dict[str, Any]) -> ExtractedRow:
    return ExtractedRow(
        party_name=str(item.get('party', '')),
        votes_text=str(item.get('votes', '0')),
        source_page=item.get('source_page'),
        candidate_name=item.get('candidate_name'),
        rank_hint=item.get('rank_hint'),
        raw_line=item.get('raw_line'),
        match_score=item.get('match_score'),
    )


def parse_markdown_rows(markdown_text: str, template_rows_for_doc: Sequence[TemplateRow], source_page: int) -> List[ExtractedRow]:
    allowed_parties = [row.party_name for row in template_rows_for_doc]
    parsed_rows: List[ExtractedRow] = []
    for line in iter_markdown_lines(markdown_text):
        segments = [segment for segment in line.split(' | ') if normalize_text(segment)]
        if not segments:
            continue
        if len(segments) == 1 and any(keyword in line for keyword in SUMMARY_KEYWORDS):
            continue
        vote_candidates = [extract_vote_candidate(segment) for segment in reversed(segments)]
        vote_candidates.append(extract_vote_candidate(line))
        votes = max(vote_candidates, key=len, default='')
        if not votes:
            continue
        matched_party = None
        matched_score = 0.0
        for target in segments + [line]:
            candidate_party, score = best_party_match(target, allowed_parties, threshold=0.56)
            if candidate_party is not None and score > matched_score:
                matched_party = candidate_party
                matched_score = score
        if matched_party is None:
            continue
        parsed_rows.append(
            ExtractedRow(
                party_name=matched_party,
                votes_text=votes,
                source_page=source_page,
                candidate_name=extract_candidate_name(line, matched_party, votes),
                rank_hint=extract_rank_hint(line),
                raw_line=line,
                match_score=round(matched_score, 4),
            )
        )
    return parsed_rows


def classify_page_content(
    markdown_text: str,
    template_rows_for_doc: Sequence[TemplateRow],
    page_number: int,
    parsed_rows: Optional[Sequence[ExtractedRow]] = None,
    image_table_score: float = 0.0,
    detected_table_regions: Optional[Sequence[Sequence[int]]] = None,
) -> Dict[str, Any]:
    rows = list(parsed_rows or [])
    table_region_count = len(detected_table_regions or [])
    unique_party_hits = len({normalize_party_key(row.party_name) for row in rows if normalize_party_key(row.party_name)})
    row_hits = len(rows)
    numeric_hits = sum(1 for row in rows if normalize_vote_text(row.votes_text))
    summary_hits = sum(1 for keyword in SUMMARY_KEYWORDS if keyword in normalize_markdown_text(markdown_text))
    has_table_bars = '|' in markdown_text
    signal_score = min(
        1.0,
        (image_table_score * 0.55) +
        (table_region_count * 0.10) +
        (unique_party_hits * 0.16) +
        (row_hits * 0.12) +
        (numeric_hits * 0.06) +
        (0.08 if has_table_bars else 0.0) +
        (0.06 if page_number in {1, 2, 3, 4} else 0.0),
    )
    signal_score = round(signal_score, 4)
    has_image_table = table_region_count > 0 or image_table_score >= 0.22
    if has_image_table and (row_hits >= 1 or unique_party_hits >= 1 or numeric_hits >= 1):
        page_type = 'table'
    elif has_image_table:
        page_type = 'table_candidate'
    elif row_hits >= 2 or (unique_party_hits >= 1 and numeric_hits >= 2):
        page_type = 'mixed'
    elif page_number == 1:
        page_type = 'header'
    elif summary_hits >= 2:
        page_type = 'summary'
    else:
        page_type = 'other'
    use_for_results = page_type in {'table', 'mixed'} and row_hits > 0
    return {
        'page_type': page_type,
        'signal_score': signal_score,
        'party_hits': unique_party_hits,
        'row_hits': row_hits,
        'numeric_hits': numeric_hits,
        'summary_hits': summary_hits,
        'table_region_count': table_region_count,
        'image_table_score': round(image_table_score, 4),
        'use_for_results': use_for_results,
    }


def map_rows_to_template(doc_id: str, extracted_rows: Sequence[ExtractedRow], template_rows_for_doc: Sequence[TemplateRow]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    by_party = template_lookup(template_rows_for_doc)
    allowed_parties = [row.party_name for row in template_rows_for_doc]
    votes_by_party: Dict[str, str] = {row.party_name: '0' for row in template_rows_for_doc}
    diagnostics: List[Dict[str, Any]] = []
    for extracted in extracted_rows:
        raw_party = normalize_text(extracted.party_name)
        if not raw_party:
            diagnostics.append({'doc_id': doc_id, 'issue': 'missing_party_name', 'row': serialize_extracted_row(extracted)})
            continue
        direct = by_party.get(normalize_party_key(raw_party))
        if direct is not None:
            matched_party = direct.party_name
            score = 1.0
        else:
            matched_party, score = best_party_match(raw_party, allowed_parties)
            if matched_party is None:
                diagnostics.append({'doc_id': doc_id, 'issue': 'unmatched_party_name', 'score': score, 'row': serialize_extracted_row(extracted)})
                continue
        votes_by_party[matched_party] = choose_best_vote(votes_by_party.get(matched_party), extracted.votes_text)
        diagnostics.append({
            'doc_id': doc_id,
            'issue': 'mapped',
            'observed_party': raw_party,
            'matched_party': matched_party,
            'score': round(score, 4),
            'votes': votes_by_party[matched_party],
            'source_page': extracted.source_page,
        })
    submission_rows: List[Dict[str, Any]] = []
    for row in template_rows_for_doc:
        submission_rows.append({
            'id': row.id,
            'doc_id': row.doc_id,
            'row_num': row.row_num,
            'party_name': row.party_name,
            'votes': coerce_votes(votes_by_party.get(row.party_name, '0')),
        })
    return submission_rows, diagnostics


def build_result_items(doc_id: str, submission_rows: Sequence[Dict[str, Any]], extracted_rows: Sequence[ExtractedRow]) -> List[Dict[str, Any]]:
    doc_meta = parse_doc_id(doc_id)
    best_candidate_name: Dict[str, str] = {}
    best_rank_hint: Dict[str, int] = {}
    for row in extracted_rows:
        key = normalize_party_key(row.party_name)
        if row.candidate_name and len(row.candidate_name) > len(best_candidate_name.get(key, '')):
            best_candidate_name[key] = row.candidate_name
        if row.rank_hint is not None and key not in best_rank_hint:
            best_rank_hint[key] = row.rank_hint
    results: List[Dict[str, Any]] = []
    for row in submission_rows:
        key = normalize_party_key(row['party_name'])
        item: Dict[str, Any] = {
            'number': best_rank_hint.get(key, int(row['row_num'])),
            'party': row['party_name'],
            'votes': int(coerce_votes(row.get('votes', '0'))),
        }
        if doc_meta['type'] == 'constituency':
            item['name'] = best_candidate_name.get(key)
        results.append(item)
    return results


def write_submission(rows: Sequence[Dict[str, Any]], output_path: Path) -> None:
    fieldnames = ['id', 'votes']
    with output_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({'id': row['id'], 'votes': coerce_votes(row.get('votes', '0'))})


## 5. EasyOCR Engine

**วัตถุประสงค์:** นิยามฟังก์ชันทั้งหมดที่เกี่ยวข้องกับ EasyOCR ตั้งแต่การโหลด model, ตรวจจับตารางด้วย OpenCV ไปจนถึงการแปลง OCR result เป็น markdown

**กลุ่มฟังก์ชันหลัก:**

**Lazy Loading & Rate Control**
- `ensure_vision_libs()`: โหลด `cv2` และ `numpy` เมื่อจำเป็น ติดตั้งอัตโนมัติหายังไม่มี
- `ensure_easyocr_reader()`: สร้าง `easyocr.Reader` singleton (โหลดครั้งเดียว) รองรับภาษาไทย + อังกฤษ
- `before_ocr_request()`: จัดการ rate limiting และ cooldown pause ระหว่าง OCR calls
- `clear_torch_cuda_cache()`: ล้าง CUDA memory cache ก่อนแต่ละ readtext call

**Table Region Detection (OpenCV)**
- `detect_table_regions(image_path)`: ตรวจหาตำแหน่งตารางในภาพด้วย morphological operations (horizontal + vertical line detection) คืน bounding boxes และ diagnostics
  - resize ภาพให้ขนาดไม่เกิน `TABLE_DETECTION_MAX_SIDE` ก่อนประมวลผล
  - มี fallback สำหรับหน้าแรก 4 หน้าเมื่อตรวจไม่พบตารางชัดเจน
- `_merge_boxes()` / `_boxes_touch()`: รวม bounding box ที่อยู่ใกล้กัน
- `_expand_box()`: เพิ่ม padding รอบ bounding box

**OCR Execution**
- `readtext_with_fallbacks()`: รัน EasyOCR พร้อม fallback ลด canvas_size อัตโนมัติเมื่อ CUDA out of memory
- `_prepare_crop_for_ocr()`: upscale + Gaussian blur crop ก่อนส่งให้ OCR เพื่อเพิ่มความแม่นยำ
- `ocr_image_to_markdown()`: จัดการ cache, crop ตาม table regions, รัน EasyOCR และบันทึก cache
- `is_retryable_easyocr_error()`: ตรวจว่า error เกิดจาก timeout / memory / connection (retry ได้)

**Layout Reconstruction**
- `easyocr_results_to_markdown()`: จัด EasyOCR bounding box results เป็นบรรทัด โดยใช้ y-coordinate grouping และแทรก ` | ` เมื่อมี horizontal gap ระหว่าง text block (จำลองโครงสร้างตาราง)

**Page Scan Orchestration**
- `scan_page()`: ฟังก์ชันหลักที่รวมทุกขั้นตอน: detect table -> OCR -> parse rows -> classify page -> คืน dict ผลลัพธ์ครบถ้วน


In [ ]:
LAST_EASYOCR_CALL_AT = 0.0
OCR_REQUEST_COUNT = 0
_EASYOCR_READER = None
_CV2 = None
_NP = None


def get_page_artifact_stem(image_path: Path) -> str:
    doc_id, page_number = parse_page_record(image_path)
    return f'{doc_id}_{page_number}'


def get_page_cache_path(image_path: Path) -> Path:
    return OCR_CACHE_DIR / f'{get_page_artifact_stem(image_path)}.md'


def get_cut_image_path(image_path: Path) -> Path:
    return IMAGE_CUT_DIR / f'{get_page_artifact_stem(image_path)}.png'


def ensure_vision_libs():
    global _CV2, _NP
    if _CV2 is not None and _NP is not None:
        return _CV2, _NP
    try:
        import cv2 as cv2_module
    except ImportError:
        print('Installing opencv-python-headless ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'opencv-python-headless'])
        import cv2 as cv2_module
    try:
        import numpy as np_module
    except ImportError:
        print('Installing numpy ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numpy'])
        import numpy as np_module
    _CV2 = cv2_module
    _NP = np_module
    return _CV2, _NP


def ensure_easyocr_reader():
    global _EASYOCR_READER
    if _EASYOCR_READER is not None:
        return _EASYOCR_READER
    try:
        import easyocr
    except ImportError:
        print('Installing easyocr ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'easyocr'])
        import easyocr
    _EASYOCR_READER = easyocr.Reader(EASYOCR_LANGUAGES, gpu=EASYOCR_USE_GPU)
    return _EASYOCR_READER


def is_retryable_easyocr_error(exc: Exception) -> bool:
    message = str(exc).lower()
    keywords = ['timeout', 'temporarily', 'connection', 'cuda', 'memory', 'resource']
    return any(keyword in message for keyword in keywords)


def before_ocr_request() -> None:
    global LAST_EASYOCR_CALL_AT, OCR_REQUEST_COUNT
    if PAUSE_EVERY_PAGES > 0 and PAUSE_SECONDS > 0 and OCR_REQUEST_COUNT > 0 and OCR_REQUEST_COUNT % PAUSE_EVERY_PAGES == 0:
        print(f'Cooling down for {PAUSE_SECONDS} seconds after {OCR_REQUEST_COUNT} EasyOCR pages ...')
        time.sleep(PAUSE_SECONDS)
    wait_seconds = MIN_INTERVAL_SECONDS - (time.time() - LAST_EASYOCR_CALL_AT)
    if wait_seconds > 0:
        time.sleep(wait_seconds)


def clear_torch_cuda_cache() -> None:
    try:
        import gc
        import torch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def readtext_with_fallbacks(reader: Any, image: Any) -> Sequence[Any]:
    settings = [
        {'canvas_size': EASYOCR_CANVAS_SIZE, 'mag_ratio': EASYOCR_MAG_RATIO},
        {'canvas_size': 1920, 'mag_ratio': 1.0},
        {'canvas_size': 1600, 'mag_ratio': 1.0},
    ]
    last_exc: Optional[Exception] = None
    for opts in settings:
        try:
            clear_torch_cuda_cache()
            return reader.readtext(
                image,
                detail=1,
                paragraph=False,
                contrast_ths=0.05,
                adjust_contrast=0.7,
                text_threshold=0.45,
                low_text=0.2,
                link_threshold=0.25,
                canvas_size=opts['canvas_size'],
                mag_ratio=opts['mag_ratio'],
            )
        except Exception as exc:
            last_exc = exc
            if 'out of memory' not in str(exc).lower():
                raise
    raise RuntimeError(f'EasyOCR region failed after memory fallbacks: {last_exc}') from last_exc


def _boxes_touch(box_a: Tuple[int, int, int, int], box_b: Tuple[int, int, int, int], gap: int) -> bool:
    ax, ay, aw, ah = box_a
    bx, by, bw, bh = box_b
    return not (
        (ax + aw + gap) < bx or
        (bx + bw + gap) < ax or
        (ay + ah + gap) < by or
        (by + bh + gap) < ay
    )


def _merge_boxes(boxes: Sequence[Tuple[int, int, int, int]], gap: int) -> List[Tuple[int, int, int, int]]:
    pending = [tuple(box) for box in boxes]
    merged: List[Tuple[int, int, int, int]] = []
    while pending:
        x, y, w, h = pending.pop(0)
        changed = True
        while changed:
            changed = False
            remaining: List[Tuple[int, int, int, int]] = []
            for other in pending:
                if _boxes_touch((x, y, w, h), other, gap):
                    ox, oy, ow, oh = other
                    x2 = max(x + w, ox + ow)
                    y2 = max(y + h, oy + oh)
                    x = min(x, ox)
                    y = min(y, oy)
                    w = x2 - x
                    h = y2 - y
                    changed = True
                else:
                    remaining.append(other)
            pending = remaining
        merged.append((x, y, w, h))
    return sorted(merged, key=lambda item: (item[1], item[0]))


def _expand_box(box: Tuple[int, int, int, int], image_shape: Tuple[int, int], padding: int) -> Tuple[int, int, int, int]:
    height, width = image_shape
    x, y, w, h = box
    x1 = max(0, x - padding)
    y1 = max(0, y - padding)
    x2 = min(width, x + w + padding)
    y2 = min(height, y + h + padding)
    return x1, y1, x2 - x1, y2 - y1


def detect_table_regions(image_path: Path, page_number: Optional[int] = None) -> Tuple[List[List[int]], Dict[str, Any]]:
    cv2, _ = ensure_vision_libs()
    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f'Unable to load image: {image_path}')
    orig_h, orig_w = gray.shape[:2]
    resize_ratio = 1.0
    if max(orig_h, orig_w) > TABLE_DETECTION_MAX_SIDE:
        resize_ratio = TABLE_DETECTION_MAX_SIDE / float(max(orig_h, orig_w))
        working = cv2.resize(gray, (max(1, int(orig_w * resize_ratio)), max(1, int(orig_h * resize_ratio))), interpolation=cv2.INTER_AREA)
    else:
        working = gray
    work_h, work_w = working.shape[:2]
    inverted = cv2.bitwise_not(working)
    binary = cv2.adaptiveThreshold(inverted, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 35, -5)
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(20, work_w // 30), 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(20, work_h // 45)))
    horizontal = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel, iterations=1)
    vertical = cv2.morphologyEx(binary, cv2.MORPH_OPEN, vertical_kernel, iterations=1)
    line_mask = cv2.add(horizontal, vertical)
    line_mask = cv2.dilate(line_mask, cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7)), iterations=1)
    contours_info = cv2.findContours(line_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = contours_info[0] if len(contours_info) == 2 else contours_info[1]
    candidate_boxes: List[Tuple[int, int, int, int]] = []
    candidate_scores: List[float] = []
    image_area = float(max(1, work_h * work_w))
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = float(max(1, w * h))
        area_ratio = area / image_area
        width_ratio = w / float(max(1, work_w))
        height_ratio = h / float(max(1, work_h))
        region = line_mask[y:y + h, x:x + w]
        density = cv2.countNonZero(region) / area
        strong_wide = width_ratio >= TABLE_MIN_WIDTH_RATIO and height_ratio >= TABLE_MIN_HEIGHT_RATIO
        compact_bottom = (y / float(max(1, work_h)) >= 0.40) and width_ratio >= (TABLE_MIN_WIDTH_RATIO * 0.7) and height_ratio >= (TABLE_MIN_HEIGHT_RATIO * 0.7)
        if area_ratio < TABLE_MIN_REGION_AREA_RATIO:
            continue
        if density < TABLE_MIN_LINE_DENSITY and not (strong_wide or compact_bottom):
            continue
        if not strong_wide and not compact_bottom:
            continue
        candidate_boxes.append((x, y, w, h))
        candidate_scores.append(min(1.0, area_ratio * 10.0 + density * 2.0))
    fallback_used = False
    if not candidate_boxes and page_number in {1, 2, 3, 4}:
        band_start = int(work_h * 0.45)
        band = line_mask[band_start:, :]
        band_density = cv2.countNonZero(band) / float(max(1, band.size))
        non_zero = cv2.findNonZero(band)
        if non_zero is not None and band_density >= TABLE_FALLBACK_DENSITY:
            x, y, w, h = cv2.boundingRect(non_zero)
            y += band_start
            width_ratio = w / float(max(1, work_w))
            height_ratio = h / float(max(1, work_h))
            if width_ratio >= 0.20 and height_ratio >= 0.03:
                candidate_boxes.append((x, y, w, h))
                candidate_scores.append(min(1.0, band_density * 40.0 + width_ratio * 0.3 + height_ratio * 0.8))
                fallback_used = True
    merge_gap = max(12, int(min(work_h, work_w) * 0.02))
    merged_boxes = _merge_boxes(candidate_boxes, gap=merge_gap)
    scaled_boxes: List[List[int]] = []
    for x, y, w, h in merged_boxes:
        if resize_ratio != 1.0:
            x = int(round(x / resize_ratio))
            y = int(round(y / resize_ratio))
            w = int(round(w / resize_ratio))
            h = int(round(h / resize_ratio))
        ex, ey, ew, eh = _expand_box((x, y, w, h), (orig_h, orig_w), TABLE_REGION_PADDING)
        scaled_boxes.append([int(ex), int(ey), int(ew), int(eh)])
    best_score = max(candidate_scores, default=0.0)
    diagnostics = {
        'table_score': round(min(1.0, best_score + (0.12 * len(scaled_boxes))), 4),
        'fallback_used': fallback_used,
        'raw_region_count': len(candidate_boxes),
        'merged_region_count': len(scaled_boxes),
    }
    return scaled_boxes, diagnostics


def _line_sort_key(item: Tuple[float, float, float, float, str]) -> Tuple[float, float]:
    return (item[0], item[1])


def easyocr_results_to_markdown(results: Sequence[Any]) -> str:
    line_items: List[Tuple[float, float, float, float, str]] = []
    heights: List[float] = []
    for entry in results:
        if not isinstance(entry, (list, tuple)) or len(entry) < 2:
            continue
        bbox = entry[0]
        text = normalize_text(entry[1])
        confidence = float(entry[2]) if len(entry) >= 3 else 1.0
        if not text:
            continue
        if confidence < EASYOCR_CONFIDENCE_THRESHOLD and len(normalize_vote_text(text)) < 2:
            continue
        xs = [float(point[0]) for point in bbox]
        ys = [float(point[1]) for point in bbox]
        left = min(xs)
        right = max(xs)
        center_y = sum(ys) / len(ys)
        height = max(ys) - min(ys)
        heights.append(height)
        line_items.append((center_y, left, right, height, text))
    if not line_items:
        return ''
    line_items.sort(key=_line_sort_key)
    median_height = sorted(heights)[len(heights) // 2] if heights else 18.0
    y_threshold = max(10.0, median_height * 0.7)
    grouped_lines: List[List[Tuple[float, float, float, float, str]]] = []
    line_centers: List[float] = []
    for item in line_items:
        center_y = item[0]
        if not grouped_lines or abs(center_y - line_centers[-1]) > y_threshold:
            grouped_lines.append([item])
            line_centers.append(center_y)
        else:
            grouped_lines[-1].append(item)
            line_centers[-1] = (line_centers[-1] + center_y) / 2.0
    markdown_lines: List[str] = []
    gap_threshold = max(18.0, median_height * 1.2)
    for line in grouped_lines:
        ordered = sorted(line, key=lambda item: item[1])
        line_text = ordered[0][4]
        prev_right = ordered[0][2]
        for item in ordered[1:]:
            separator = ' | ' if (item[1] - prev_right) > gap_threshold else ' '
            line_text += separator + item[4]
            prev_right = item[2]
        markdown_lines.append(line_text)
    return normalize_markdown_text('\n'.join(markdown_lines))


def _prepare_crop_for_ocr(gray_crop: Any) -> Any:
    cv2, _ = ensure_vision_libs()
    if gray_crop is None or getattr(gray_crop, 'size', 0) == 0:
        return gray_crop
    crop = gray_crop
    height, width = crop.shape[:2]
    if min(height, width) < 700:
        scale = min(1.8, max(1.15, 720.0 / float(max(1, min(height, width)))))
        crop = cv2.resize(crop, (max(1, int(width * scale)), max(1, int(height * scale))), interpolation=cv2.INTER_CUBIC)
    crop = cv2.GaussianBlur(crop, (3, 3), 0)
    return crop


def _union_table_regions(table_regions: Sequence[Sequence[int]], image_shape: Tuple[int, int]) -> Tuple[int, int, int, int]:
    height, width = image_shape
    if not table_regions:
        return 0, 0, width, height
    x1 = min(int(box[0]) for box in table_regions)
    y1 = min(int(box[1]) for box in table_regions)
    x2 = max(int(box[0]) + int(box[2]) for box in table_regions)
    y2 = max(int(box[1]) + int(box[3]) for box in table_regions)
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(width, x2)
    y2 = min(height, y2)
    return x1, y1, max(1, x2 - x1), max(1, y2 - y1)


def prepare_cut_image_for_page(image_path: Path, page_number: Optional[int] = None) -> Dict[str, Any]:
    cv2, _ = ensure_vision_libs()
    if page_number is None:
        _, page_number = parse_page_record(image_path)
    table_regions, table_region_diagnostics = detect_table_regions(image_path, page_number=page_number)
    cut_path = get_cut_image_path(image_path)
    color = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if color is None:
        raise FileNotFoundError(f'Unable to load image: {image_path}')
    height, width = color.shape[:2]
    if table_regions:
        x, y, w, h = _union_table_regions(table_regions, (height, width))
        crop = color[y:y + h, x:x + w]
        cut_mode = 'table_crop'
    else:
        x, y, w, h = 0, 0, width, height
        crop = color
        cut_mode = 'full_image'
    if REBUILD_IMAGE_CUT or FORCE_RERUN_OCR or not cut_path.exists():
        ok = cv2.imwrite(str(cut_path), crop)
        if not ok:
            raise RuntimeError(f'Failed to write cut image: {cut_path}')
    return {
        'cut_image_path': cut_path,
        'cut_mode': cut_mode,
        'cut_box': [int(x), int(y), int(w), int(h)],
        'table_regions': table_regions,
        'table_region_diagnostics': table_region_diagnostics,
    }


def ocr_image_to_markdown(source_image_path: Path, input_image_path: Path) -> Tuple[str, Path]:
    global LAST_EASYOCR_CALL_AT, OCR_REQUEST_COUNT
    cv2, _ = ensure_vision_libs()
    cache_path = get_page_cache_path(source_image_path)
    if cache_path.exists() and not FORCE_RERUN_OCR:
        return normalize_markdown_text(cache_path.read_text(encoding='utf-8')), cache_path
    reader = ensure_easyocr_reader()
    gray = cv2.imread(str(input_image_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f'Unable to load image: {input_image_path}')
    last_exc: Optional[Exception] = None
    doc_id, page_number = parse_page_record(source_image_path)
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            before_ocr_request()
            OCR_REQUEST_COUNT += 1
            LAST_EASYOCR_CALL_AT = time.time()
            prepared = _prepare_crop_for_ocr(gray)
            results = readtext_with_fallbacks(reader, prepared)
            markdown = normalize_markdown_text(easyocr_results_to_markdown(results))
            cache_path.write_text(markdown, encoding='utf-8')
            return markdown, cache_path
        except Exception as exc:
            last_exc = exc
            if attempt >= MAX_RETRIES or not is_retryable_easyocr_error(exc):
                raise
            sleep_seconds = RETRY_BASE_SECONDS * (2 ** (attempt - 1))
            print(f'Retryable EasyOCR error on {doc_id} page {page_number}: {exc}. Sleeping {sleep_seconds}s.')
            time.sleep(sleep_seconds)
    raise RuntimeError(f'EasyOCR failed for {source_image_path.name}') from last_exc


def scan_page(image_path: Path, template_rows_for_doc: Sequence[TemplateRow]) -> Dict[str, Any]:
    doc_id, page_number = parse_page_record(image_path)
    cut_info = prepare_cut_image_for_page(image_path, page_number=page_number)
    markdown_text, cache_path = ocr_image_to_markdown(image_path, cut_info['cut_image_path'])
    parsed_rows = parse_markdown_rows(markdown_text, template_rows_for_doc, source_page=page_number) if cut_info['table_regions'] else []
    classification = classify_page_content(
        markdown_text,
        template_rows_for_doc,
        page_number,
        parsed_rows=parsed_rows,
        image_table_score=cut_info['table_region_diagnostics']['table_score'],
        detected_table_regions=cut_info['table_regions'],
    )
    return {
        'doc_id': doc_id,
        'page_number': page_number,
        'source_image_path': str(image_path),
        'cut_image_path': str(cut_info['cut_image_path']),
        'cut_mode': cut_info['cut_mode'],
        'cut_box': cut_info['cut_box'],
        'page_type': classification['page_type'],
        'signal_score': classification['signal_score'],
        'party_hits': classification['party_hits'],
        'row_hits': classification['row_hits'],
        'numeric_hits': classification['numeric_hits'],
        'summary_hits': classification['summary_hits'],
        'use_for_results': classification['use_for_results'],
        'image_table_score': classification['image_table_score'],
        'table_regions': cut_info['table_regions'],
        'ocr_region_count': len(cut_info['table_regions']) if cut_info['table_regions'] else 1,
        'ocr_scope': cut_info['cut_mode'],
        'table_region_diagnostics': cut_info['table_region_diagnostics'],
        'ocr_cache_path': str(cache_path),
        'ocr_markdown': markdown_text if STORE_MARKDOWN_IN_LABELS else None,
        'parsed_rows': [serialize_extracted_row(row) for row in parsed_rows],
    }



## 5A. Prepare image_cut PNGs




In [ ]:
def get_active_image_paths_for_cut(limit_docs: Optional[int] = LIMIT_DOCS) -> List[Path]:
    active_docs = doc_ids if limit_docs is None else doc_ids[:limit_docs]
    paths: List[Path] = []
    for doc_id in active_docs:
        paths.extend(document_index[doc_id])
    return paths


def prepare_all_cut_images(limit_docs: Optional[int] = LIMIT_DOCS) -> None:
    image_paths = get_active_image_paths_for_cut(limit_docs=limit_docs)
    for image_path in tqdm(image_paths, desc='prepare image_cut'):
        prepare_cut_image_for_page(image_path)
    print(f'Prepare image_cut complete: {len(image_paths)} PNG files in {IMAGE_CUT_DIR}')


if RUN_PREPARE_IMAGE_CUT:
    prepare_all_cut_images()
else:
    print(f'Set RUN_PREPARE_IMAGE_CUT = True to save one cropped/full PNG per page in {IMAGE_CUT_DIR}')



## 6. Scan Stage: OCR ทุกหน้าและบันทึก JSON Label

**วัตถุประสงค์:** วนซ้ำผ่านไฟล์ภาพทุกไฟล์, เรียก `scan_page()` ต่อแต่ละหน้า และบันทึกผลลัพธ์เป็น JSON file หนึ่งไฟล์ต่อหน้าใน `data/labels/`

**ฟังก์ชันที่นิยามใน cell นี้:**

- `get_page_label_path(image_path)`: คืน path ของ JSON label สำหรับหน้านั้น เช่น `data/labels/constituency_10_1_2.json`
- `build_page_results()`: จัดกลุ่ม extracted rows ตามพรรค เลือก best vote ต่อพรรค และสร้าง result list
- `build_page_label_payload()`: สร้าง JSON payload ครบถ้วน รวม province metadata, page classification, OCR markdown และ diagnostics
- `save_page_label()` / `load_page_label()`: เขียน/อ่าน JSON label file
- `scan_image_to_label()`: ตรวจสอบ cache ก่อน ถ้ายังไม่มี (หรือ `RESCAN_EXISTING_LABELS=True`) จึงรัน scan และบันทึก
- `scan_all_pages()`: วนซ้ำ image paths ทั้งหมดพร้อม progress bar และรัน `scan_image_to_label()` ต่อแต่ละไฟล์

**เงื่อนไขการรัน:** cell นี้จะเรียก `scan_all_pages()` ก็ต่อเมื่อ `RUN_SCAN_TO_LABELS = True` ใน Config

**โครงสร้าง JSON label ที่บันทึก:**
```json
{
  "province_name": "...",
  "doc_id": "constituency_10_1",
  "page_number": 2,
  "page_type": "table",
  "use_for_results": true,
  "results": [{"number": 1, "party": "...", "votes": 1234}],
  "diagnostics": {...},
  "ocr_markdown": "..."
}
```


In [ ]:
def get_page_label_stem(image_path: Path) -> str:
    return get_page_artifact_stem(image_path)


def get_page_label_path(image_path: Path) -> Path:
    return LABEL_DIR / f'{get_page_label_stem(image_path)}.json'


def build_page_results(doc_id: str, extracted_rows: Sequence[ExtractedRow]) -> List[Dict[str, Any]]:
    doc_meta = parse_doc_id(doc_id)
    grouped: Dict[str, ExtractedRow] = {}
    for row in extracted_rows:
        key = normalize_party_key(row.party_name)
        existing = grouped.get(key)
        if existing is None:
            grouped[key] = row
            continue
        grouped[key] = ExtractedRow(
            party_name=existing.party_name,
            votes_text=choose_best_vote(existing.votes_text, row.votes_text),
            source_page=existing.source_page,
            candidate_name=existing.candidate_name or row.candidate_name,
            rank_hint=existing.rank_hint if existing.rank_hint is not None else row.rank_hint,
            raw_line=existing.raw_line or row.raw_line,
            match_score=max(existing.match_score or 0.0, row.match_score or 0.0),
        )
    ordered_rows = sorted(grouped.values(), key=lambda row: (row.rank_hint is None, row.rank_hint if row.rank_hint is not None else 9999, row.party_name))
    results: List[Dict[str, Any]] = []
    for idx, row in enumerate(ordered_rows, 1):
        item: Dict[str, Any] = {
            'number': row.rank_hint if row.rank_hint is not None else idx,
            'party': row.party_name,
            'votes': int(coerce_votes(row.votes_text)),
        }
        if doc_meta['type'] == 'constituency':
            item['name'] = row.candidate_name
        results.append(item)
    return results


def build_page_label_payload(image_path: Path, page_info: Dict[str, Any]) -> Dict[str, Any]:
    doc_id, page_number = parse_page_record(image_path)
    doc_meta = parse_doc_id(doc_id)
    extracted_rows = [deserialize_extracted_row(item) for item in page_info.get('parsed_rows', [])]
    payload: Dict[str, Any] = {
        'province_name': known_province_names.get(doc_meta['province_code']),
        'constituency_number': doc_meta['constituency_number'],
        'type': doc_meta['type'],
        'summary_by_section': {},
        'results': build_page_results(doc_id, extracted_rows) if page_info.get('use_for_results') else [],
        'province_code': doc_meta['province_code'],
        'doc_id': doc_id,
        'page_number': page_number,
        'page_type': page_info['page_type'],
        'use_for_results': page_info['use_for_results'],
        'source_image_path': page_info['source_image_path'],
        'cut_image_path': page_info['cut_image_path'],
        'ocr_cache_path': page_info['ocr_cache_path'],
        'ocr_markdown': page_info['ocr_markdown'],
        'diagnostics': {
            'signal_score': page_info['signal_score'],
            'party_hits': page_info['party_hits'],
            'row_hits': page_info['row_hits'],
            'numeric_hits': page_info['numeric_hits'],
            'summary_hits': page_info.get('summary_hits', 0),
            'image_table_score': page_info.get('image_table_score', 0.0),
            'ocr_scope': page_info.get('ocr_scope', 'table_crop'),
            'ocr_region_count': page_info.get('ocr_region_count', 0),
            'cut_mode': page_info.get('cut_mode', 'table_crop'),
            'cut_box': page_info.get('cut_box', []),
            'detected_table_regions': page_info.get('table_regions', []),
            'table_region_diagnostics': page_info.get('table_region_diagnostics', {}),
        },
    }
    return payload


def save_page_label(image_path: Path, payload: Dict[str, Any]) -> Path:
    output_path = get_page_label_path(image_path)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return output_path


def load_page_label(image_path: Path) -> Dict[str, Any]:
    return json.loads(get_page_label_path(image_path).read_text(encoding='utf-8'))


def get_active_image_paths(limit_docs: Optional[int] = LIMIT_DOCS) -> List[Path]:
    active_docs = doc_ids if limit_docs is None else doc_ids[:limit_docs]
    paths: List[Path] = []
    for doc_id in active_docs:
        paths.extend(document_index[doc_id])
    return paths


def scan_image_to_label(image_path: Path) -> Dict[str, Any]:
    label_path = get_page_label_path(image_path)
    if label_path.exists() and not RESCAN_EXISTING_LABELS:
        return load_page_label(image_path)
    doc_id, _ = parse_page_record(image_path)
    page_info = scan_page(image_path, template_by_doc[doc_id])
    payload = build_page_label_payload(image_path, page_info)
    save_page_label(image_path, payload)
    return payload


def scan_all_pages(limit_docs: Optional[int] = LIMIT_DOCS) -> None:
    image_paths = get_active_image_paths(limit_docs=limit_docs)
    for image_path in tqdm(image_paths, desc='scan pages'):
        scan_image_to_label(image_path)
    print(f'Scan complete: {len(image_paths)} page labels in {LABEL_DIR}')


if RUN_SCAN_TO_LABELS:
    scan_all_pages()
else:
    print(f'Set RUN_SCAN_TO_LABELS = True to OCR cropped/full pages from {IMAGE_CUT_DIR} and save one JSON per sheet in {LABEL_DIR}')



## 7. Build submission.csv from Page Labels

**วัตถุประสงค์:** รวม JSON label จากทุกหน้าของแต่ละเอกสาร, จับคู่คะแนนเสียงกับ scaffold template และเขียนไฟล์ `data/submission.csv` ในรูปแบบ `id,votes` ที่ใช้ submit

**ฟังก์ชันที่นิยามใน cell นี้:**

- `load_submission_scaffold()`: อ่าน `submission_template_v3.csv` ซึ่งมีคอลัมน์ `id`, `votes`, และ `party_name_*` ครบ 10,053 แถว
- `validate_submission_scaffold()`: ตรวจสอบว่า scaffold มีครบ 10,053 แถวและ id ไม่ซ้ำ
- `validate_submission_rows()`: ตรวจสอบ output ก่อน write ว่า row count ตรง, id ไม่ซ้ำ และ votes เป็นตัวเลขทั้งหมด
- `build_doc_label_index()`: สร้าง index จาก doc_id ไปยัง list ของ label paths ทุกหน้าของเอกสารนั้น
- `find_vote_for_party_in_payload()`: ค้นหา votes ของพรรคที่ระบุใน results ของ label payload
- `find_vote_for_party_in_paths()`: วนหาใน candidate paths ทั้งหมด (primary label ก่อน ถ้าไม่พบจึงดูหน้าอื่นในเอกสาร)
- `build_submission_for_rows()`: ประมวลผล scaffold rows และเก็บ stats (primary_match, doc_page_match, missing ฯลฯ)
- `build_submission_in_chunks()`: ฟังก์ชันหลัก รัน pipeline ทั้งหมด, บันทึก partial CSV ระหว่างทาง, validate และ write ผลสุดท้าย

**เงื่อนไขการรัน:** cell นี้จะเรียก `build_submission_in_chunks()` ก็ต่อเมื่อ `RUN_BUILD_SUBMISSION = True` ใน Config

**Fault tolerance:** เขียน `submission.partial.csv` ระหว่าง process และลบทิ้งเมื่อสำเร็จ เพื่อกู้คืนได้หาก kernel หยุดกลางคัน


In [ ]:
def get_scaffold_party_column(fieldnames: Sequence[str]) -> str:
    for field in fieldnames:
        if field.startswith('party_name'):
            return field
    raise ValueError('Could not find party_name column in submission scaffold')


def load_submission_scaffold(path: Path = SUBMISSION_SCAFFOLD_PATH) -> Tuple[List[Dict[str, str]], List[str]]:
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        fieldnames = list(reader.fieldnames or [])
        if 'id' not in fieldnames or 'votes' not in fieldnames:
            raise ValueError(f'Submission scaffold must contain id and votes columns: {path}')
        rows = [dict(row) for row in reader]
    return rows, fieldnames


def validate_submission_scaffold(scaffold_rows: Sequence[Dict[str, str]]) -> None:
    scaffold_ids = [str(row.get('id', '')).strip() for row in scaffold_rows]
    if len(scaffold_rows) != 10053:
        raise ValueError(f'Submission scaffold row count mismatch: expected 10053, found {len(scaffold_rows)}')
    if len(set(scaffold_ids)) != len(scaffold_ids):
        raise ValueError('Submission scaffold has duplicate ids')


def validate_submission_rows(rows: Sequence[Dict[str, str]], expected_ids: Sequence[str]) -> None:
    if len(rows) != len(expected_ids):
        raise ValueError(f'Submission row count mismatch: expected {len(expected_ids)}, found {len(rows)}')
    observed_ids = [str(row.get('id', '')).strip() for row in rows]
    if len(set(observed_ids)) != len(observed_ids):
        raise ValueError('Submission contains duplicate ids')
    missing_ids = sorted(set(expected_ids) - set(observed_ids))
    extra_ids = sorted(set(observed_ids) - set(expected_ids))
    if missing_ids or extra_ids:
        raise ValueError(
            f'Submission id mismatch | missing={len(missing_ids)} extra={len(extra_ids)} '
            f"examples_missing={', '.join(missing_ids[:3])} examples_extra={', '.join(extra_ids[:3])}"
        )
    bad_votes = [row['id'] for row in rows if not re.fullmatch(r'\d+', str(row.get('votes', '')))]
    if bad_votes:
        preview = ', '.join(bad_votes[:5])
        raise ValueError(f'Submission contains non-numeric votes. Examples: {preview}')


def write_submission_rows(rows: Sequence[Dict[str, str]], output_path: Path) -> None:
    with output_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['id', 'votes'])
        writer.writeheader()
        for row in rows:
            writer.writerow({'id': row['id'], 'votes': coerce_votes(row.get('votes', '0'))})


def get_label_path_for_submission_id(row_id: str) -> Path:
    return LABEL_DIR / f'{row_id}.json'


def load_label_payload(path: Path) -> Optional[Dict[str, Any]]:
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))


def get_doc_id_from_submission_id(row_id: str) -> str:
    return row_id.rsplit('_', 1)[0] if '_' in row_id else row_id


def build_doc_label_index(label_dir: Path = LABEL_DIR) -> Dict[str, List[Path]]:
    index: Dict[str, List[Path]] = defaultdict(list)
    for path in sorted(label_dir.glob('*.json')):
        stem = path.stem
        doc_id = get_doc_id_from_submission_id(stem)
        index[doc_id].append(path)
    return index


def iter_candidate_label_paths(row_id: str, doc_label_index: Dict[str, List[Path]]) -> List[Path]:
    primary = get_label_path_for_submission_id(row_id)
    doc_id = get_doc_id_from_submission_id(row_id)
    paths: List[Path] = []
    if primary.exists():
        paths.append(primary)
    for path in doc_label_index.get(doc_id, []):
        if path != primary:
            paths.append(path)
    return paths


def find_vote_for_party_in_payload(target_party: str, payload: Optional[Dict[str, Any]]) -> str:
    if not payload:
        return '0'
    results = payload.get('results', []) or []
    if not results:
        return '0'
    target_key = normalize_party_key(target_party)
    for item in results:
        if normalize_party_key(item.get('party')) == target_key:
            return coerce_votes(item.get('votes', '0'))
    return '0'


def find_vote_for_party_in_paths(target_party: str, candidate_paths: Sequence[Path]) -> Tuple[str, Optional[Path], str]:
    # guard against empty list
    if not candidate_paths:
        return '0', None, 'missing'

    primary_path = candidate_paths[0]
    best_vote = '0'
    matched_path: Optional[Path] = None
    match_scope = 'missing'
    saw_results = False

    for path in candidate_paths:
        payload = load_label_payload(path)
        if not payload:
            continue
        results = payload.get('results', []) or []
        if not results:
            continue
        saw_results = True
        vote = find_vote_for_party_in_payload(target_party, payload)
        if vote != '0':
            best_vote = choose_best_vote(best_vote, vote)
            matched_path = path
            if path == primary_path:
                match_scope = 'primary'
                break
            else:
                match_scope = 'doc_page'

    if matched_path is not None:
        return best_vote, matched_path, match_scope
    if saw_results:
        return '0', None, 'party_missing'
    return '0', None, 'empty_or_no_table'


def build_submission_for_rows(scaffold_rows_subset: Sequence[Dict[str, str]], fieldnames: Sequence[str], doc_label_index: Dict[str, List[Path]]) -> Tuple[List[Dict[str, str]], Dict[str, int]]:
    """Build submission rows for a subset of scaffold rows.

    Returns the list of submission rows and a small stats dict.
    """
    party_column = get_scaffold_party_column(fieldnames)
    submission_rows: List[Dict[str, str]] = []
    stats = {
        'doc_has_labels': 0,
        'doc_missing': 0,
        'primary_label_match': 0,
        'doc_page_match': 0,
        'empty_or_no_table': 0,
        'party_missing': 0,
        'missing': 0,
    }

    for index, row in enumerate(scaffold_rows_subset, 1):
        row_id = str(row.get('id', '')).strip()
        party_name = str(row.get(party_column, '')).strip()
        candidate_paths = iter_candidate_label_paths(row_id, doc_label_index)

        if not candidate_paths:
            votes = '0'
            stats['doc_missing'] += 1
        else:
            stats['doc_has_labels'] += 1
            votes, matched_path, match_scope = find_vote_for_party_in_paths(party_name, candidate_paths)
            if match_scope == 'primary':
                stats['primary_label_match'] += 1
            elif match_scope == 'doc_page':
                stats['doc_page_match'] += 1
            elif match_scope == 'empty_or_no_table':
                stats['empty_or_no_table'] += 1
            elif match_scope == 'party_missing':
                stats['party_missing'] += 1
            else:
                stats['missing'] += 1

        submission_rows.append({'id': row_id, 'votes': votes})

    return submission_rows, stats


def build_submission_in_chunks(chunk_size: int = 10053) -> List[Dict[str, str]]:
    """Process the full scaffold in chunks (default 10053 rows) and produce a final submission list.

    For each scaffold row like `constituency_10_1_1,ประชาธิปัตย์,0` we look into the
    corresponding `data/labels/constituency_10_1_1.json` and other pages in the
    same document; when a table exists we match the party and fill its votes;
    when no table exists the vote remains 0.
    """
    scaffold_rows, fieldnames = load_submission_scaffold(SUBMISSION_SCAFFOLD_PATH)
    validate_submission_scaffold(scaffold_rows)
    expected_ids = [str(row.get('id', '')).strip() for row in scaffold_rows]
    doc_label_index = build_doc_label_index(LABEL_DIR)

    all_submission_rows: List[Dict[str, str]] = []
    aggregate_stats: Dict[str, int] = defaultdict(int)

    total = len(scaffold_rows)
    for start in range(0, total, chunk_size):
        end = min(start + chunk_size, total)
        chunk = scaffold_rows[start:end]
        print(f'Processing rows {start+1}-{end} (chunk size {chunk_size})...')
        chunk_rows, stats = build_submission_for_rows(chunk, fieldnames, doc_label_index)
        for k, v in stats.items():
            aggregate_stats[k] += v
        all_submission_rows.extend(chunk_rows)
        write_submission_rows(all_submission_rows, PARTIAL_OUTPUT_PATH)

    validate_submission_rows(all_submission_rows, expected_ids)
    write_submission_rows(all_submission_rows, OUTPUT_PATH)
    if PARTIAL_OUTPUT_PATH.exists():
        PARTIAL_OUTPUT_PATH.unlink()

    print(
        f"doc_has_labels={aggregate_stats['doc_has_labels']} doc_missing={aggregate_stats['doc_missing']} "
        f"primary_label_match={aggregate_stats['primary_label_match']} doc_page_match={aggregate_stats['doc_page_match']} "
        f"empty_or_no_table={aggregate_stats['empty_or_no_table']} party_missing={aggregate_stats['party_missing']} "
        f"missing={aggregate_stats['missing']}"
    )
    return all_submission_rows


if RUN_BUILD_SUBMISSION:
    # run chunked builder with default chunk_size 10053
    submission_rows = build_submission_in_chunks(chunk_size=10053)
    print(f'Wrote {len(submission_rows)} rows to {OUTPUT_PATH}')
else:
    print(f'Set RUN_BUILD_SUBMISSION = True to build {OUTPUT_PATH.name} from one-sheet JSON labels in {LABEL_DIR}')

## 8. Export Labels List CSV

**วัตถุประสงค์:** สร้างไฟล์ `data/labels_list.csv` ซึ่งเป็นรายชื่อ stem ของ JSON label ทุกไฟล์ที่มีอยู่ใน `data/labels/`

**วิธีใช้:** รัน cell นี้หลังจาก scan stage เสร็จสิ้น ไฟล์ที่ได้มีหนึ่งคอลัมน์ชื่อ `filename` บันทึก stem ของ label แต่ละไฟล์ (ไม่รวม `.json`)

ไฟล์นี้ใช้เป็น input ให้ cell ถัดไปสำหรับ filter submission rows ที่มี label จริง


In [ ]:
# Write all JSON filenames in `data/labels` to `data/labels_list.csv`
# Single-column CSV with header `filename`.

def write_labels_csv(label_dir: Path = LABEL_DIR, out_path: Path = DATA_ROOT / 'labels_list.csv') -> None:
    paths = sorted(label_dir.glob('*.json'))
    with out_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['filename'])
        for p in paths:
            writer.writerow([p.stem])
    print(f'Wrote {len(paths)} filenames to {out_path}.')

# Run the writer now (comment out the next line to skip execution)
write_labels_csv()

## 9. Filter Submission by Labels List

**วัตถุประสงค์:** อ่าน `data/labels_list.csv` แล้วตั้งค่า `votes = 0` สำหรับทุก row ใน `submission.csv` ที่ `id` ไม่ปรากฏใน labels list

**วิธีใช้:** รัน cell นี้หลังจากสร้าง `labels_list.csv` และ `submission.csv` แล้ว เพื่อป้องกันไม่ให้ row ที่ไม่มี label ส่งค่า votes ที่ไม่ได้มาจาก OCR จริง

**หมายเหตุ:** cell นี้เขียนทับ `submission.csv` โดยตรง ควร backup ไฟล์ก่อนหากต้องการเก็บค่าเดิม


In [ ]:
# Map `labels_list.csv` to `submission.csv` — keep votes for matching ids, zero others
# Reads `data/labels_list.csv` (one filename per row, header `filename`) and `data/submission.csv`
# Overwrites `data/submission.csv` such that rows whose `id` is NOT in labels_list get `votes` set to '0'.

from pathlib import Path
import csv

DATA_DIR = Path("data")
LABELS_LIST = DATA_DIR / "labels_list.csv"
SUBMISSION = DATA_DIR / "submission.csv"

if not LABELS_LIST.exists():
    print(f"{LABELS_LIST} not found. Create it first (e.g., run labels generator).")
else:
    # load labels set (filenames without .json)
    with LABELS_LIST.open("r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        header = next(reader, None)
        labels = {row[0].strip() for row in reader if row}

    if not SUBMISSION.exists():
        print(f"{SUBMISSION} not found.")
    else:
        with SUBMISSION.open("r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames
            rows = list(reader)

        kept = 0
        zeroed = 0
        for r in rows:
            _id = r.get("id") or r.get("Id") or r.get("ID")
            if _id is None:
                continue
            if _id in labels:
                kept += 1
            else:
                r["votes"] = "0"
                zeroed += 1

        # write back
        with SUBMISSION.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)

        print(f"Processed {len(rows)} rows: kept={kept}, zeroed={zeroed} — wrote {SUBMISSION}")


## 10. Count Non-Zero Votes

**วัตถุประสงค์:** ตรวจสอบคุณภาพ submission โดยนับจำนวน row ที่มีค่า `votes` ไม่เท่ากับ 0

**วิธีใช้:** รัน cell นี้เพื่อดูสัดส่วนเอกสารที่ OCR สำเร็จ หากตัวเลขต่ำผิดปกติให้ตรวจสอบ scan stage หรือ table detection

ฟังก์ชัน `count_nonzero_votes()` รองรับคอลัมน์ `votes`, `Votes` และ `VOTES` รวมถึง handle non-numeric value อย่างปลอดภัย


In [ ]:
# Count non-zero votes in `data/submission.csv`
from pathlib import Path
import csv

def count_nonzero_votes(submission_path=Path("data/submission.csv")):
    p = Path(submission_path)
    if not p.exists():
        print(f"{p} not found")
        return 0
    with p.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    total = len(rows)
    nonzero = 0
    for r in rows:
        v = r.get("votes") or r.get("Votes") or r.get("VOTES") or r.get("votes ")
        if v is None or str(v).strip() == "":
            continue
        try:
            if int(float(str(v).strip())) != 0:
                nonzero += 1
        except Exception:
            try:
                digits = ''.join(ch for ch in str(v) if ch.isdigit())
                if digits and int(digits) != 0:
                    nonzero += 1
            except Exception:
                pass

    print(f"Total rows: {total} — non-zero votes: {nonzero}")
    return nonzero

# Run the check now
count_nonzero_votes()


## 11. Find Images Without Page Number in Filename

**วัตถุประสงค์:** ค้นหาและบันทึกรายชื่อไฟล์ภาพใน `data/images/` ที่ชื่อไม่มีคำว่า `page` ซึ่งหมายความว่าเป็นเอกสารหน้าเดียว

**วิธีใช้:** รัน cell นี้เพื่อตรวจสอบว่ามีไฟล์ภาพใดที่ไม่ตรงรูปแบบ `<doc_id>_page<N>.png` ผลลัพธ์บันทึกเป็น `data/images_without_page.csv`

ไฟล์ CSV นี้จะถูกใช้ใน cell ถัดไปเพื่อสร้าง label ID สำหรับเอกสารหน้าเดียว (โดย append `_1` ต่อท้ายชื่อ)


In [ ]:
# List images in `data/images` that do NOT contain 'page' in the filename
from pathlib import Path
import csv

IMG_DIR = Path("data/images")
OUT_CSV = Path("data/images_without_page.csv")

if not IMG_DIR.exists():
    print(f"{IMG_DIR} not found")
else:
    files = [p for p in IMG_DIR.rglob("*") if p.is_file()]
    total = len(files)
    names = [p.stem for p in files]
    no_page = [n for n in names if "page" not in n.lower()]

    print(f"Total image files: {total}")
    print(f"Images without 'page' in filename: {len(no_page)}")
    if no_page:
        print("--- Listing ---")
        for n in no_page:
            print(n)

    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["filename"])
        for n in no_page:
            writer.writerow([n])
    print(f"Wrote {len(no_page)} filenames to {OUT_CSV}")


## 12. Add Title Column to Images-Without-Page CSV

**วัตถุประสงค์:** เพิ่มคอลัมน์ `Title` ใน `data/images_without_page.csv` โดยใช้ค่า `<filename>_1` เพื่อสร้าง label ID สำหรับเอกสารหน้าเดียว

**วิธีใช้:** รัน cell นี้หลังจาก cell 11 (images without page) เสร็จแล้ว ค่าใน column `Title` จะถูกนำไปใช้ใน cell ถัดไปเพื่อ zero votes

**ตัวอย่าง:** ชื่อไฟล์ `constituency_10_1` -> Title = `constituency_10_1_1`


In [ ]:
# Add `Title` column to `data/images_without_page.csv` (append "_1" to each filename)
from pathlib import Path
import csv

CSV_PATH = Path("data/images_without_page.csv")
if not CSV_PATH.exists():
    print(f"{CSV_PATH} not found. Run the image-listing cell first.")
else:
    with CSV_PATH.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        fieldnames = list(reader.fieldnames) if reader.fieldnames else []

    if not rows:
        print(f"No rows in {CSV_PATH}")
    else:
        # determine filename column (default to 'filename')
        filename_col = None
        for col in fieldnames:
            if col.lower() == "filename":
                filename_col = col
                break
        if filename_col is None:
            filename_col = fieldnames[0]

        new_col = "Title"
        for r in rows:
            orig = (r.get(filename_col) or "").strip()
            r[new_col] = orig + "_1"

        if new_col not in fieldnames:
            fieldnames.append(new_col)

        with CSV_PATH.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)

        print(f"Updated {CSV_PATH}: wrote {len(rows)} rows with new column '{new_col}'")


## 13. Zero Votes for Single-Page Documents

**วัตถุประสงค์:** ตั้งค่า `votes = 0` ใน `submission.csv` สำหรับ ID ที่อยู่ใน column `setzero` ของ `images_without_page.csv` เพื่อแก้ปัญหาเอกสารหน้าเดียวที่อาจถูก OCR ผิดพลาด

**วิธีใช้:** รัน cell นี้ต่อจาก cell 12 (add setzero column) ค่า votes ที่ไม่ใช่ 0 ในแถวที่ตรงกับ setzero IDs จะถูกตั้งเป็น 0

**หมายเหตุ:** cell นี้เขียนทับ `submission.csv` โดยตรง


In [ ]:
# Zero `votes` in `data/submission.csv` for IDs listed in `setzero` column of `data/images_without_page.csv`
from pathlib import Path
import csv

LABELS_CSV = Path("data/images_without_page.csv")
SUBMISSION_CSV = Path("data/submission.csv")

if not LABELS_CSV.exists():
    print(f"{LABELS_CSV} not found. Run the images-listing cell first.")
else:
    with LABELS_CSV.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    setzero_col = None
    if rows:
        for col in reader.fieldnames:
            if col.lower() == "Front":
                setzero_col = col
                break
        if setzero_col is None:
            # fallback: look for any column named like setzero or use last column
            for col in reader.fieldnames:
                if "Front" in col.lower():
                    setzero_col = col
                    break
        if setzero_col is None:
            setzero_col = reader.fieldnames[-1]

    set_ids = { (r.get(setzero_col) or "").strip() for r in rows if r.get(setzero_col) }
    print(f"Loaded {len(set_ids)} IDs from {LABELS_CSV} (column: {setzero_col})")

    if not SUBMISSION_CSV.exists():
        print(f"{SUBMISSION_CSV} not found")
    else:
        with SUBMISSION_CSV.open("r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames
            submission_rows = list(reader)

        zeroed = 0
        for r in submission_rows:
            _id = r.get("id") or r.get("Id") or r.get("ID")
            if _id and _id in set_ids:
                if r.get("votes") != "0":
                    r["votes"] = "0"
                    zeroed += 1

        with SUBMISSION_CSV.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(submission_rows)

        print(f"Updated {SUBMISSION_CSV}: set votes to 0 for {zeroed} rows")


## 14. Sort labels_list.csv by Natural Order

**วัตถุประสงค์:** เรียงลำดับรายการใน `data/labels_list.csv` ด้วย natural sort (ตัวเลขเรียงตามค่า ไม่ใช่ lexicographic) เพื่อให้ดูและ debug ง่ายขึ้น

**วิธีใช้:** รัน cell นี้หลังจาก `write_labels_csv()` สร้างไฟล์แล้ว ฟังก์ชัน `natural_key()` จะแบ่ง filename ออกเป็น text segment และ numeric segment แล้วเรียงให้ถูกต้อง

**ตัวอย่างการเรียง:** `constituency_1_1`, `constituency_2_1`, `constituency_10_1` (ไม่ใช่ `1`, `10`, `2` แบบ alphabetical)


In [ ]:
# Sort filenames in `data/labels_list.csv` using a natural alphanumeric order
# Reads the CSV (header `filename`), sorts rows naturally (so numbers sort numerically), and overwrites the CSV.
from pathlib import Path
import csv
import re

LABELS_CSV = Path("data/labels_list.csv")

def natural_key(s):
    parts = re.split(r"(\d+)", s)
    key = []
    for p in parts:
        if p.isdigit():
            key.append(int(p))
        else:
            key.append(p.lower())
    return key

if not LABELS_CSV.exists():
    print(f"{LABELS_CSV} not found. Create it first.")
else:
    with LABELS_CSV.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames or ["filename"]
        rows = list(reader)

    # determine filename column
    filename_col = fieldnames[0]
    for col in fieldnames:
        if col.lower() == "filename":
            filename_col = col
            break

    # sort rows by natural key of the filename value
    rows_sorted = sorted(rows, key=lambda r: natural_key((r.get(filename_col) or "").strip()))

    with LABELS_CSV.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows_sorted)

    sample = [ (r.get(filename_col) or "") for r in rows_sorted[:20] ]
    print(f"Sorted {len(rows_sorted)} filenames and wrote {LABELS_CSV}. Sample: {sample}")


In [ ]:
# Extract trailing digit from filenames in `data/labels_list.csv` into a new `number` column
from pathlib import Path
import csv

LABELS_CSV = Path("data/labels_list.csv")
if not LABELS_CSV.exists():
    print(f"{LABELS_CSV} not found")
else:
    with LABELS_CSV.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        fieldnames = reader.fieldnames or ["filename"]

    if not rows:
        print(f"No rows in {LABELS_CSV}")
    else:
        # determine filename column
        filename_col = None
        for col in fieldnames:
            if col.lower() == "filename":
                filename_col = col
                break
        if filename_col is None:
            filename_col = fieldnames[0]

        new_col = "number"
        for r in rows:
            val = (r.get(filename_col) or "").strip()
            if val and val[-1].isdigit():
                r[new_col] = val[-1]
            else:
                r[new_col] = ""

        if new_col not in fieldnames:
            fieldnames.append(new_col)

        with LABELS_CSV.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)

        sample = rows[:10]
        print(f"Wrote {len(rows)} rows to {LABELS_CSV} with new column '{new_col}'. Sample:")
        for r in sample:
            print(r.get(filename_col), r.get(new_col))


In [ ]:
# Scan `labels_list.csv` starting from line 3; when the next row's `number` == '1',
# take the current row's `filename` and set it as `Back` in `data/images_without_page.csv`.
# If the Title match exists in images_without_page.csv, update that row; otherwise append a new row.
from pathlib import Path
import csv

LABELS_CSV = Path('data/labels_list.csv')
IMG_NOPAGE_CSV = Path('data/images_without_page.csv')

if not LABELS_CSV.exists():
    print(f"{LABELS_CSV} not found. Create it first.")
elif not IMG_NOPAGE_CSV.exists():
    print(f"{IMG_NOPAGE_CSV} not found. Run the images-listing cell first.")
else:
    with LABELS_CSV.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        labels = list(reader)

    if not labels:
        print('No rows in labels_list.csv')
    else:
        # Build list of candidate 'back' filenames
        candidates = []
        # start from line 3 -> index 2 (0-based)
        start_idx = 2
        for i in range(start_idx, len(labels)-1):
            cur = labels[i]
            nxt = labels[i+1]
            cur_num = (cur.get('number') or '').strip()
            nxt_num = (nxt.get('number') or '').strip()
            # condition: if next row's number == '1', take current filename
            if nxt_num == '1':
                fname = (cur.get('filename') or '').strip()
                if fname:
                    candidates.append(fname)

        candidates = list(dict.fromkeys(candidates))  # unique preserve order

        # Load images_without_page.csv
        with IMG_NOPAGE_CSV.open('r', encoding='utf-8', newline='') as f:
            reader = csv.DictReader(f)
            rows = list(reader)
            fieldnames = list(reader.fieldnames) if reader.fieldnames else ['filename','Title']

        if 'Back' not in fieldnames:
            fieldnames.append('Back')

        updated = 0
        appended = 0
        examples = []
        # index rows by Title and filename for quick lookup
        title_index = { (r.get('Title') or '').strip(): r for r in rows }
        filename_index = { (r.get('filename') or '').strip(): r for r in rows }

        for c in candidates:
            matched = False
            if c in title_index:
                title_index[c]['Back'] = c
                updated += 1
                matched = True
                if len(examples) < 299:
                    examples.append(('updated', c))
            elif c in filename_index:
                filename_index[c]['Back'] = c
                updated += 1
                matched = True
                if len(examples) < 299:
                    examples.append(('updated-filename', c))
            else:
                # append a new row with filename and Title set to c, Back=c
                new_row = {fn: '' for fn in fieldnames}
                # guess filename base as c without last suffix if underscores
                new_row['filename'] = c
                new_row['Title'] = c
                new_row['Back'] = c
                rows.append(new_row)
                appended += 1
                if len(examples) < 299:
                    examples.append(('appended', c))

        # write back
        with IMG_NOPAGE_CSV.open('w', encoding='utf-8', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)

        print(f"Processed candidates: {len(candidates)} — updated: {updated}, appended: {appended}")
        if examples:
            print('Examples (action, filename):')
            for e in examples:
                print(e)
